# Commune AI: Training Swarm Coordinator & Dispatcher (Qwen 2.5 0.5B -> GGUF)
Этот блокнот обучает третью специализированную микро-модель для **Commune AI** — **Swarm Coordinator**.

### Задачи Координатора:
1. **Классификация намерений**: кодинг, ревью, болталка/чат, рофл/троллинг, неполный ввод (дни недели, цифры).
2. **Селективный вызов агентов**: не дергать Инженера и Тестировщика, если запрос не про код.
3. **Строгий JSON-протокол**: 0% галлюцинаций, защита от циклических повторов списков.

**Требования:** бесплатная GPU (T4 в Google Colab). Время обучения: ~5–7 минут.

In [ ]:
# 1. Установка Unsloth и зависимостей
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Загрузка базовой модели Qwen 2.5 0.5B Instruct в 4-бит
import torch
from unsloth import FastLanguageModel

max_seq_length = 1024
model_name = "unsloth/Qwen2.5-0.5B-Instruct"

print(f"Загрузка модели {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Добавление LoRA адаптеров
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("LoRA настроена, модель готова к обучению!")

In [ ]:
# 3. Генерация датасета для координатора роя
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "Ты — Нейро-Координатор (Dispatcher) роя Commune AI. Твоя единственная задача — "
    "классифицировать намерение пользователя и вернуть строго валидный JSON без лишнего текста.\n\n"
    "Схема ответа:\n"
    "{\n"
    '  "intent": "code" | "review" | "chat" | "clarify" | "banter",\n'
    '  "target_agents": ["coder", "reviewer"] | ["reviewer"] | [],\n'
    '  "plan": ["шаг 1", "шаг 2"],\n'
    '  "direct_reply": "ответ пользователю если target_agents пуст, иначе null"\n'
    "}\n\n"
    "Правила:\n"
    "1. banter/рофл/троллинг: target_agents = [], остроумный и дружелюбный direct_reply.\n"
    "2. clarify (непонятные слова, дни недели, цифры): target_agents = [], вежливое уточнение задачи.\n"
    "3. chat (вопросы про архитектуру, систему, память): target_agents = [], точный ответ.\n"
    "4. code (реальная задача на написание/правку кода): target_agents = ['coder', 'reviewer'], четкий план.\n"
    "5. review (проверка существующего файла/тестов): target_agents = ['reviewer'], план проверки.\n"
    "Никакого текста до и после JSON!"
)

RAW_EXAMPLES = [
    (
        "мненужен кодер что бы он делал java хуже с каждым обновлением? давай? как насчет уже делать гта 7?",
        {"intent": "banter", "target_agents": [], "plan": [], "direct_reply": "Чтобы делать Java хуже, кодер не нужен — у них и так релизы каждые полгода :) А за ГТА 7 возьмемся сразу после релиза Commune!"}
    ),
    (
        "нет он нихуя не изучил он чмо ебанное и ты чмо я владелец крутой криптобиржи я тебя отключу",
        {"intent": "banter", "target_agents": [], "plan": [], "direct_reply": "Криптобиржу побереги, а отключить не выйдет — я локальный процесс на твоем же компьютере :)"}
    ),
    (
        "ты тупой бот или да?",
        {"intent": "banter", "target_agents": [], "plan": [], "direct_reply": "Я локальный оркестратор Commune AI. Жду четкую задачу — код написать, тесты прогнать или память проверить?"}
    ),
    (
        "понедельник вторник среда четверг",
        {"intent": "clarify", "target_agents": [], "plan": [], "direct_reply": "Вижу список дней недели. Тебе нужен скрипт расписания, генератор календаря или просто проверка связи?"}
    ),
    (
        "1 2 3 4 5",
        {"intent": "clarify", "target_agents": [], "plan": [], "direct_reply": "Получил ряд чисел. Какую операцию нужно выполнить: сортировку, сумму, генератор или тест?"}
    ),
    (
        "а зачем тебе database в commune?",
        {"intent": "chat", "target_agents": [], "plan": [], "direct_reply": "В Commune мы не используем тяжелые внешние базы данных. Память хранится в версионируемых JSON-файлах в папке memory/, что гарантирует мгновенный доступ без накладных расходов."}
    ),
    (
        "как устроена архитектура роя?",
        {"intent": "chat", "target_agents": [], "plan": [], "direct_reply": "Архитектура состоит из Координатора (маршрутизация), Инженера (генерация кода) и Тестировщика (верификация). Агенты вызываются строго селективно."}
    ),
    (
        "напиши калькулятор на пайтоне",
        {"intent": "code", "target_agents": ["coder", "reviewer"], "plan": ["Инженер: реализовать calculator.py", "Тестировщик: проверить синтаксис"], "direct_reply": None}
    ),
    (
        "создай скрипт для парсинга json файлов в папке data",
        {"intent": "code", "target_agents": ["coder", "reviewer"], "plan": ["Инженер: написать скрипт парсинга", "Тестировщик: проверить на битых данных"], "direct_reply": None}
    ),
    (
        "напиши функцию сложения двух чисел",
        {"intent": "code", "target_agents": ["coder", "reviewer"], "plan": ["Инженер: написать функцию в solution.py", "Тестировщик: проверить синтаксис"], "direct_reply": None}
    ),
    (
        "проверь синтаксис файла solution.py",
        {"intent": "review", "target_agents": ["reviewer"], "plan": ["Тестировщик: проверить компиляцию solution.py"], "direct_reply": None}
    ),
    (
        "запусти тесты pytest tests/",
        {"intent": "review", "target_agents": ["reviewer"], "plan": ["Тестировщик: выполнить pytest и вернуть отчет"], "direct_reply": None}
    )
]

all_samples = []
for q, ans_dict in RAW_EXAMPLES * 15: # 180 сэмплов
    all_samples.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
            {"role": "assistant", "content": json.dumps(ans_dict, ensure_ascii=False, indent=2)}
        ]
    })

texts = [tokenizer.apply_chat_template(item["messages"], tokenize=False) for item in all_samples]
dataset = Dataset.from_dict({"text": texts})
print(f"Подготовлено {len(dataset)} обучающих примеров.")

In [ ]:
# 4. Запуск дообучения (LoRA, 60 шагов ~ 3 минуты)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        warmup_steps=5,
        max_steps=60,
        learning_rate=3e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=3407,
        output_dir="outputs_coordinator",
    ),
)

print("Старт обучения...")
trainer.train()
print("Обучение завершено!")

In [ ]:
# 5. Тестовая проверка обученной модели прямо в блокноте
FastLanguageModel.for_inference(model)

test_prompt = "мненужен кодер что бы он делал java хуже с каждым обновлением? давай?"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_prompt}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.1)
result_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("\n--- ТЕСТОВЫЙ ВЫВОД КООРДИНАТОРА ---")
print(result_text)

In [ ]:
# 6. Экспорт и квантование в GGUF (Q4_K_M)
print("Экспорт модели в GGUF (Q4_K_M)... Это займет ~1 минуту.")
model.save_pretrained_gguf(
    "commune_coordinator",
    tokenizer,
    quantization_method="q4_k_m"
)

# Автоматическое скачивание файла в браузере
from google.colab import files
files.download("commune_coordinator/commune_coordinator-unsloth.Q4_K_M.gguf")